# Retrieval Eval
Compares **dense-only** vs **hybrid (BM25 + dense + rerank)** across Hit Rate@k and MRR@k.

In [ ]:
import csv
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, str(Path.cwd().parent))
from retrieve import get_collection
from hybrid_retrieve import HybridRetriever

load_dotenv(Path.cwd().parent.parent / ".env")

EMBED_MODEL = "text-embedding-3-small"
COST_PER_1M = 0.02
CHROMA_PATH = str(Path.cwd().parent / "chroma_db")
CHUNKS_PATH = str(Path.cwd().parent / "data" / "chunks.json")
KS = [1, 3, 5, 10]
MAX_K = max(KS)
EMBED_BATCH = 2048

client = OpenAI()
collection = get_collection(CHROMA_PATH)
print(f"Collection: {collection.count()} chunks")

In [8]:
dataset = []
with open("dataset.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        dataset.append({"question": row["question"], "chunk_id": row["chunk_id"].strip(), "answer": row["answer"]})
print(f"Loaded {len(dataset)} eval pairs")

Loaded 222 eval pairs


## Dense-only eval

In [ ]:
# Batched embed (max 2048 per call) + parallel Chroma queries
t0 = time.time()
questions = [r["question"] for r in dataset]
embeddings, tokens = [], 0
for i in range(0, len(questions), EMBED_BATCH):
    resp = client.embeddings.create(model=EMBED_MODEL, input=questions[i:i+EMBED_BATCH], encoding_format="float")
    embeddings += [item.embedding for item in sorted(resp.data, key=lambda x: x.index)]
    tokens += resp.usage.total_tokens
print(f"Embedded {len(questions)} queries in {time.time()-t0:.1f}s  cost: ${tokens/1_000_000*COST_PER_1M:.6f}")

def _query_chroma(args):
    idx, row, vec = args
    hits = collection.query(query_embeddings=[vec], n_results=MAX_K, include=["metadatas", "documents", "distances"])
    ranked_ids = hits["ids"][0]
    rank = next((j + 1 for j, cid in enumerate(ranked_ids) if cid == row["chunk_id"]), None)
    return idx, {
        "question": row["question"],
        "expected_chunk_id": row["chunk_id"],
        "expected_answer": row["answer"],
        "rank": rank,
        "rr": 1 / rank if rank else 0.0,
        "ranked_ids": ranked_ids,
        "ranked_meta": hits["metadatas"][0],
        "ranked_docs": hits["documents"][0],
        "ranked_scores": hits["distances"][0],
    }

t0 = time.time()
dense_results = [None] * len(dataset)
with ThreadPoolExecutor(max_workers=16) as pool:
    futures = [pool.submit(_query_chroma, (i, row, vec)) for i, (row, vec) in enumerate(zip(dataset, embeddings))]
    for done in as_completed(futures):
        idx, result = done.result()
        dense_results[idx] = result
print(f"Chroma queries done in {time.time()-t0:.1f}s")

## Hybrid eval (BM25 + dense + cross-encoder rerank)
First run downloads the cross-encoder model (~90MB).

In [11]:
retriever = HybridRetriever(CHUNKS_PATH, CHROMA_PATH, openai_client=client)
print("Ready.")

Building BM25 index...
Loading cross-encoder (downloads on first run ~90MB)...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Ready.


In [ ]:
# batch_retrieve: parallel embed+BM25+RRF, then one batched cross-encoder call
t0 = time.time()
all_ranked = retriever.batch_retrieve([r["question"] for r in dataset], n_results=MAX_K, k_candidates=10)
print(f"Done in {time.time()-t0:.1f}s")

hybrid_results = []
for row, ranked in zip(dataset, all_ranked):
    ranked_ids = [r["chunk_id"] for r in ranked]
    rank = next((j + 1 for j, cid in enumerate(ranked_ids) if cid == row["chunk_id"]), None)
    hybrid_results.append({
        "question": row["question"],
        "expected_chunk_id": row["chunk_id"],
        "expected_answer": row["answer"],
        "rank": rank,
        "rr": 1 / rank if rank else 0.0,
        "ranked": ranked,
    })

## Metrics comparison

In [13]:
rows = []
for k in KS:
    d_hit = sum(1 for r in dense_results if r["rank"] and r["rank"] <= k) / len(dense_results)
    d_mrr = sum(r["rr"] for r in dense_results if r["rank"] and r["rank"] <= k) / len(dense_results)
    h_hit = sum(1 for r in hybrid_results if r["rank"] and r["rank"] <= k) / len(hybrid_results)
    h_mrr = sum(r["rr"] for r in hybrid_results if r["rank"] and r["rank"] <= k) / len(hybrid_results)
    rows.append({"k": k, "Dense Hit Rate": round(d_hit, 3), "Dense MRR": round(d_mrr, 3),
                 "Hybrid Hit Rate": round(h_hit, 3), "Hybrid MRR": round(h_mrr, 3)})

pd.DataFrame(rows).set_index("k")

,Dense Hit Rate,Dense MRR,Hybrid Hit Rate,Hybrid MRR
k,,,,
1,0.692,0.692,0.640,0.640
3,0.923,0.782,0.838,0.727
5,1.000,0.801,0.892,0.740
10,1.000,0.801,0.937,0.747


## Miss inspection
Queries missed by **dense** but found by **hybrid**, and vice versa.

In [ ]:
dense_miss_ids = {r["expected_chunk_id"] for r in dense_results if not r["rank"] or r["rank"] > 5}
hybrid_miss_ids = {r["expected_chunk_id"] for r in hybrid_results if not r["rank"] or r["rank"] > 5}

hybrid_rescued = dense_miss_ids - hybrid_miss_ids
hybrid_lost = hybrid_miss_ids - dense_miss_ids

print(f"Misses at k=5 — Dense: {len(dense_miss_ids)}, Hybrid: {len(hybrid_miss_ids)}")
print(f"Rescued by hybrid: {len(hybrid_rescued)}")
print(f"Lost by hybrid:    {len(hybrid_lost)}")

In [ ]:
# Drill into a miss — change `results_to_inspect` and `idx`
results_to_inspect = dense_results  # or hybrid_results
misses = [r for r in results_to_inspect if not r["rank"] or r["rank"] > 5]
idx = 0  # change to inspect different misses

miss = misses[idx]
print(f"Question: {miss['question']}")
print(f"Expected chunk_id: {miss['expected_chunk_id']}")
print(f"Expected answer: {miss['expected_answer']}")
print()

expected = collection.get(ids=[miss["expected_chunk_id"]], include=["documents", "metadatas"])
if expected["ids"]:
    print("--- Expected chunk ---")
    print(f"Title:   {expected['metadatas'][0]['article_title']}")
    print(f"Heading: {expected['metadatas'][0]['heading']}")
    print(expected["documents"][0][:500])
else:
    print("Expected chunk not in DB!")

print()
print("--- What was retrieved instead (top 5) ---")
key = "ranked" if "ranked" in miss else "ranked_ids"
top5 = miss[key][:5] if key == "ranked" else list(zip(miss["ranked_ids"][:5], miss["ranked_meta"][:5], miss["ranked_docs"][:5], miss["ranked_scores"][:5]))
if key == "ranked":
    for i, r in enumerate(top5):
        print(f"\nRank {i+1} (score={r['score']:.4f}) | {r['source_title']} | {r['chunk'][:200]}...")
else:
    for i, (cid, meta, doc, score) in enumerate(top5):
        print(f"\nRank {i+1} (score={score:.4f}) | {meta['article_title']} | {doc[:200]}...")

## Late hits — ranked > 3

In [ ]:
for label, res in [("Dense", dense_results), ("Hybrid", hybrid_results)]:
    late = [{"rank": r["rank"], "question": r["question"][:80]} for r in res if r["rank"] and r["rank"] > 3]
    print(f"\n{label} — {len(late)} late hits (rank > 3):")
    display(pd.DataFrame(late).sort_values("rank") if late else pd.DataFrame())